# Legal-BERT Clause Classifier
---

## Why Legal-BERT?

**Legal-BERT** (`nlpaueb/legal-bert-base-uncased`) is a BERT model pre-trained on legal corpora
(EU legislation, court cases, contracts). Unlike general-purpose BERT, it understands legal
terminology, clause structures, and domain-specific language patterns.

## Why Transformers After Classical ML?

Our classical baselines established strong benchmarks:
- **Logistic Regression**: F1=87.5%, Recall=81.9%
- **LinearSVC**: F1=91.3%, Recall=87.9%

However, TF-IDF + classical ML has fundamental limitations:
- Cannot capture word order or context
- Misses semantic meaning (e.g., "shall not terminate" vs "shall terminate")
- No transfer learning from legal domain knowledge

Legal-BERT addresses all of these with deep contextual embeddings.

## Why Start With One Label?

Starting with a single label ("Termination For Convenience") allows us to:
1. Validate the pipeline end-to-end before scaling
2. Debug tokenization, training, and evaluation in isolation
3. Establish transformer baseline metrics for comparison
4. Control computational cost on CPU-only hardware

---
## Phase 1 — Load & Filter Dataset

In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Load full dataset
df_full = pd.read_csv(r"C:\Users\chari\Desktop\Contract_Intelligence_AI\data\processed\clause_classification_dataset.csv")

# Filter for single label
TARGET_LABEL = "Termination For Convenience"
df = df_full[df_full['label_name'] == TARGET_LABEL].reset_index(drop=True)

print("=" * 60)
print(f"DATASET FILTERED: {TARGET_LABEL}")
print("=" * 60)
print(f"\nFull dataset shape: {df_full.shape}")
print(f"Filtered dataset shape: {df.shape}")
print(f"\nClass Distribution:")
print(df['target'].value_counts())
print(f"\nPositive ratio: {df['target'].mean():.2%}")
print(f"\nText Length Stats:")
print(df['text'].str.len().describe())

### Sample Rows

In [ ]:
pd.set_option('display.max_colwidth', 100)
df.head(10)

---
## Phase 2 — Train/Test Split

In [ ]:
from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42,
    stratify=df['target']
)

train_df = train_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print("=" * 60)
print("TRAIN/TEST SPLIT COMPLETE")
print("=" * 60)
print(f"\nTrain: {len(train_df)} samples")
print(f"Test:  {len(test_df)} samples")
print(f"\nTrain target distribution:\n{train_df['target'].value_counts()}")
print(f"\nTest target distribution:\n{test_df['target'].value_counts()}")

---
## Phase 3 — Legal-BERT Tokenization

Load the Legal-BERT tokenizer and convert text into transformer-ready input format:
- `input_ids`: token indices
- `attention_mask`: which tokens are real vs padding
- Truncation at 512 tokens (BERT max)

In [ ]:
from transformers import AutoTokenizer

MODEL_NAME = "nlpaueb/legal-bert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print("=" * 60)
print("TOKENIZER LOADED")
print("=" * 60)
print(f"\nModel: {MODEL_NAME}")
print(f"Vocab size: {tokenizer.vocab_size}")
print(f"Max length: {tokenizer.model_max_length}")

# Test tokenization on a sample
sample_text = train_df['text'].iloc[0][:200]
tokens = tokenizer(sample_text, truncation=True, padding=True, max_length=512)
print(f"\nSample tokenization:")
print(f"  Text: {sample_text[:100]}...")
print(f"  Input IDs length: {len(tokens['input_ids'])}")
print(f"  Attention mask length: {len(tokens['attention_mask'])}")

---
## Phase 4 — Build HuggingFace Dataset Objects

Convert pandas DataFrames into HuggingFace `Dataset` objects with tokenized inputs.

In [ ]:
import torch
from torch.utils.data import Dataset as TorchDataset

class ClauseDataset(TorchDataset):
    """Custom PyTorch Dataset for legal clause classification."""

    def __init__(self, texts, labels, tokenizer, max_length=512):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        label = int(self.labels[idx])

        encoding = self.tokenizer(
            text,
            truncation=True,
            padding='max_length',
            max_length=self.max_length,
            return_tensors='pt'
        )

        return {
            'input_ids': encoding['input_ids'].squeeze(0),
            'attention_mask': encoding['attention_mask'].squeeze(0),
            'labels': torch.tensor(label, dtype=torch.long)
        }

# Create datasets
train_dataset = ClauseDataset(
    texts=train_df['text'].tolist(),
    labels=train_df['target'].tolist(),
    tokenizer=tokenizer,
    max_length=512
)

test_dataset = ClauseDataset(
    texts=test_df['text'].tolist(),
    labels=test_df['target'].tolist(),
    tokenizer=tokenizer,
    max_length=512
)

print("=" * 60)
print("DATASETS CREATED")
print("=" * 60)
print(f"\nTraining samples: {len(train_dataset)}")
print(f"Test samples: {len(test_dataset)}")

# Verify a sample
sample = train_dataset[0]
print(f"\nSample verification:")
print(f"  input_ids shape: {sample['input_ids'].shape}")
print(f"  attention_mask shape: {sample['attention_mask'].shape}")
print(f"  label: {sample['labels'].item()}")

---
## Phase 5 — Load Legal-BERT for Binary Classification

Load the pre-trained Legal-BERT model and add a classification head for binary prediction (`num_labels=2`).

In [ ]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2
)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print("=" * 60)
print("LEGAL-BERT MODEL LOADED")
print("=" * 60)
print(f"\nModel: {MODEL_NAME}")
print(f"Classification: Binary (num_labels=2)")
print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

---
## Phase 6 — Training Configuration

Configure training for CPU-based laptop training:
- Small batch size (8) to fit in memory
- 2 epochs for initial experimentation
- Evaluation at the end of each epoch
- Save the best model based on F1-score

In [ ]:
from transformers import TrainingArguments, Trainer
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import os

# Output directory
output_dir = r"C:\Users\chari\Desktop\Contract_Intelligence_AI\models\legal_bert_classifier"
os.makedirs(output_dir, exist_ok=True)

def compute_metrics(eval_pred):
    """Compute evaluation metrics for the Trainer."""
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)

    return {
        'accuracy': accuracy_score(labels, predictions),
        'precision': precision_score(labels, predictions, zero_division=0),
        'recall': recall_score(labels, predictions, zero_division=0),
        'f1': f1_score(labels, predictions, zero_division=0)
    }

training_args = TrainingArguments(
    output_dir=output_dir,
    num_train_epochs=2,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    save_total_limit=2,
    seed=42,
    report_to="none",
    fp16=False,
    use_cpu=True
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics
)

print("=" * 60)
print("TRAINING CONFIGURATION")
print("=" * 60)
print(f"\nEpochs: {training_args.num_train_epochs}")
print(f"Batch size (train): {training_args.per_device_train_batch_size}")
print(f"Batch size (eval): {training_args.per_device_eval_batch_size}")
print(f"Device: CPU")
print(f"Best model metric: {training_args.metric_for_best_model}")
print(f"Output directory: {output_dir}")

---
## Phase 7 — Train Legal-BERT

Fine-tune Legal-BERT on the "Termination For Convenience" clause classification task.

> **Note**: Training on CPU will take longer than on GPU. For 510 samples with 2 epochs, expect ~10-30 minutes.

In [ ]:
import time

print("=" * 60)
print("STARTING LEGAL-BERT TRAINING")
print("=" * 60)
print()

start_time = time.time()
train_result = trainer.train()
elapsed = time.time() - start_time

print()
print("=" * 60)
print("TRAINING COMPLETE")
print("=" * 60)
minutes = int(elapsed // 60)
seconds = int(elapsed % 60)
print(f"\nTraining time: {minutes}m {seconds}s")
print(f"Training loss: {train_result.training_loss:.4f}")

---
## Phase 8 — Evaluation

Evaluate the fine-tuned Legal-BERT model on the held-out test set.

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

# Run evaluation
eval_results = trainer.evaluate()

print("=" * 60)
print("LEGAL-BERT EVALUATION RESULTS")
print("=" * 60)
print(f"\n  Accuracy:   {eval_results['eval_accuracy']:.4f}")
print(f"  Precision:  {eval_results['eval_precision']:.4f}")
print(f"  Recall:     {eval_results['eval_recall']:.4f}  <- CRITICAL for legal AI")
print(f"  F1-Score:   {eval_results['eval_f1']:.4f}  <- CRITICAL for legal AI")
print(f"  Loss:       {eval_results['eval_loss']:.4f}")

# Detailed predictions
predictions = trainer.predict(test_dataset)
y_pred = np.argmax(predictions.predictions, axis=-1)
y_true = test_df['target'].values

print("\n" + "=" * 60)
print("CLASSIFICATION REPORT")
print("=" * 60)
print(classification_report(y_true, y_pred, target_names=['Absent (0)', 'Present (1)']))

# Confusion matrix
cm = confusion_matrix(y_true, y_pred)
tn, fp, fn, tp = cm.ravel()
print(f"True Negatives:  {tn}")
print(f"False Positives: {fp}")
print(f"False Negatives: {fn}  <- CRITICAL (missed clauses)")
print(f"True Positives:  {tp}")

---
## Phase 9 — Comparison With Classical Baselines

Compare Legal-BERT performance against our classical ML baselines on the same label.

In [ ]:
# ==========================================
# COMPARE WITH CLASSICAL BASELINES
# ==========================================

# Load SVM metrics for Termination For Convenience
import json

svm_metrics_path = r"metrics.json"
lr_metrics_path = r"metrics.json"

with open(svm_metrics_path, "r") as f:
    svm_metrics = json.load(f)

with open(lr_metrics_path, "r") as f:
    lr_metrics = json.load(f)

# Extract per-label metrics for our target label
svm_label = svm_metrics['per_label_metrics'].get(TARGET_LABEL, {})
lr_label = lr_metrics['per_label_metrics'].get(TARGET_LABEL, {})

bert_metrics = {
    'accuracy': eval_results['eval_accuracy'],
    'precision': eval_results['eval_precision'],
    'recall': eval_results['eval_recall'],
    'f1_score': eval_results['eval_f1']
}

print("=" * 70)
print(f"MODEL COMPARISON: {TARGET_LABEL}")
print("=" * 70)
print()
print(f"  {'Metric':<12s} {'LR':>10s} {'SVM':>10s} {'Legal-BERT':>12s} {'Best':>10s}")
print(f"  {'-'*12} {'-'*10} {'-'*10} {'-'*12} {'-'*10}")

for metric, display in [('accuracy','Accuracy'), ('precision','Precision'),
                         ('recall','Recall'), ('f1_score','F1-Score')]:
    lr_val = lr_label.get(metric, 0)
    svm_val = svm_label.get(metric, 0)
    bert_val = bert_metrics[metric]

    best_val = max(lr_val, svm_val, bert_val)
    if best_val == bert_val:
        winner = "BERT"
    elif best_val == svm_val:
        winner = "SVM"
    else:
        winner = "LR"

    marker = " <-" if metric in ['recall', 'f1_score'] else ""
    print(f"  {display:<12s} {lr_val:>10.4f} {svm_val:>10.4f} {bert_val:>12.4f} {winner:>10s}{marker}")

---
## Phase 10 — Save Model Artifacts

In [ ]:
import json

save_dir = r"C:\Users\chari\Desktop\Contract_Intelligence_AI\models\legal_bert_classifier"
os.makedirs(save_dir, exist_ok=True)

# ==========================================
# 1. Save Model
# ==========================================
model_save_path = os.path.join(save_dir, "model")
model.save_pretrained(model_save_path)
print(f"Model saved: {model_save_path}")

# ==========================================
# 2. Save Tokenizer
# ==========================================
tokenizer_save_path = os.path.join(save_dir, "tokenizer")
tokenizer.save_pretrained(tokenizer_save_path)
print(f"Tokenizer saved: {tokenizer_save_path}")

# ==========================================
# 3. Save Metrics
# ==========================================
metrics_to_save = {
    "model": "nlpaueb/legal-bert-base-uncased",
    "task": "binary_classification",
    "target_label": TARGET_LABEL,
    "num_labels": 2,
    "training_config": {
        "epochs": int(training_args.num_train_epochs),
        "batch_size": training_args.per_device_train_batch_size,
        "max_length": 512,
        "device": "cpu"
    },
    "split": {
        "test_size": 0.2,
        "random_state": 42,
        "train_samples": len(train_df),
        "test_samples": len(test_df)
    },
    "evaluation_metrics": {
        "accuracy": round(float(eval_results['eval_accuracy']), 4),
        "precision": round(float(eval_results['eval_precision']), 4),
        "recall": round(float(eval_results['eval_recall']), 4),
        "f1_score": round(float(eval_results['eval_f1']), 4),
        "loss": round(float(eval_results['eval_loss']), 4)
    },
    "confusion_matrix": {
        "true_negatives": int(tn),
        "false_positives": int(fp),
        "false_negatives": int(fn),
        "true_positives": int(tp)
    }
}

metrics_path = os.path.join(save_dir, "metrics.json")
with open(metrics_path, "w") as f:
    json.dump(metrics_to_save, f, indent=2)
print(f"Metrics saved: {metrics_path}")

# ==========================================
# VERIFY
# ==========================================
print("\n" + "=" * 60)
print("ALL LEGAL-BERT ARTIFACTS SAVED")
print("=" * 60)
print(f"\nDirectory: {save_dir}")
for item in os.listdir(save_dir):
    full_path = os.path.join(save_dir, item)
    if os.path.isdir(full_path):
        n_files = len(os.listdir(full_path))
        print(f"  {item}/ ({n_files} files)")
    else:
        size_kb = os.path.getsize(full_path) / 1024
        print(f"  {item} ({size_kb:.1f} KB)")